# 7. The same test on real data

Notebook 1 makes its case on a dataset I wrote. That is a fair objection to it:
a simulation can be tuned until it produces the result its author wanted.

This notebook runs the identical test on two standard public benchmarks that I
did not construct, and that the fairness literature has used for years.

- **UCI Adult** (1994 US census extract, 45k rows). Used by Hardt, Price &
  Srebro (2016) among many others. Contains `sex` and `race`.
- **German Credit / Statlog** (1000 real loan applications). The protected
  attribute is not a column called `sex`. It is hidden inside `personal_status`,
  which is the point.

The result is stronger on real data than on mine.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score


def adversary_auc(df, target):
    """Cross-validated AUC for recovering `target` from `df`. 0.5 is chance."""
    cats = df.select_dtypes(include=['category', 'object']).columns.tolist()
    pre = ColumnTransformer(
        [('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cats)],
        remainder='passthrough')
    pipe = make_pipeline(pre, HistGradientBoostingClassifier(random_state=0))
    return cross_val_score(pipe, df, target, cv=5, scoring='roc_auc', n_jobs=-1).mean()

## UCI Adult

Drop `sex`, then keep dropping whatever looks like a proxy, and watch the
adversary.

In [2]:
adult = fetch_openml(data_id=1590, as_frame=True, parser='auto').frame.dropna()
is_female = (adult['sex'] == 'Female').values
X = adult.drop(columns=['sex', 'class'])

print(f'{len(adult)} rows, {is_female.mean():.1%} female')
print(f'columns available to the adversary: {list(X.columns)}\n')

removals = [
    ('sex column dropped, nothing else', []),
    ('+ relationship', ['relationship']),
    ('+ marital-status', ['relationship', 'marital-status']),
    ('+ occupation', ['relationship', 'marital-status', 'occupation']),
    ('+ hours-per-week', ['relationship', 'marital-status', 'occupation', 'hours-per-week']),
    ('+ education and workclass', ['relationship', 'marital-status', 'occupation',
                                   'hours-per-week', 'education', 'education-num', 'workclass']),
]
for label, drop in removals:
    print(f'  {label:<38} adversary AUC = {adversary_auc(X.drop(columns=drop), is_female):.3f}')
print(f'\n  {"chance":<38} adversary AUC = 0.500')

45222 rows, 32.5% female
columns available to the adversary: ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country']



  sex column dropped, nothing else       adversary AUC = 0.938


  + relationship                         adversary AUC = 0.878


  + marital-status                       adversary AUC = 0.823


  + occupation                           adversary AUC = 0.720


  + hours-per-week                       adversary AUC = 0.670


  + education and workclass              adversary AUC = 0.644

  chance                                 adversary AUC = 0.500


## Why the first drop matters so much

`relationship` takes these values:

In [3]:
print(adult['relationship'].value_counts().to_string())
print()
print(pd.crosstab(adult['relationship'], adult['sex'], normalize='index')
      .round(3).to_string())

relationship
Husband           18666
Not-in-family     11702
Own-child          6626
Unmarried          4788
Wife               2091
Other-relative     1349

sex             Female   Male
relationship                 
Husband          0.000  1.000
Not-in-family    0.462  0.538
Other-relative   0.452  0.548
Own-child        0.442  0.558
Unmarried        0.763  0.237
Wife             1.000  0.000


`Husband` and `Wife` are sex, written out in full, in a column nobody would list
as a protected attribute. This is not a subtle statistical proxy. It is the
attribute itself under a different name, and it survives any process that works
from a list of column names.

And note what happens after it is removed: the adversary drops from 0.938 to
0.878 and no further towards chance. Even the fourth removal leaves it well above
0.5, on real census data, with no help from me.

## German Credit

1000 real loan applications. There is no `sex` column at all. There is
`personal_status`.

In [4]:
german = fetch_openml(data_id=31, as_frame=True, parser='auto').frame
print(german['personal_status'].value_counts().to_string())

personal_status
male single           548
female div/dep/mar    310
male mar/wid           92
male div/sep           50


The categories encode marital status and sex together. Anyone auditing this file
for protected attributes by scanning column names finds nothing.

In [5]:
is_female_de = german['personal_status'].astype(str).str.contains('female').values
Xg = german.drop(columns=['personal_status', 'class'])

print(f'{len(german)} applications, {is_female_de.mean():.1%} female\n')
print(f'  {"personal_status dropped":<38} adversary AUC = '
      f'{adversary_auc(Xg, is_female_de):.3f}')
print(f'  {"chance":<38} adversary AUC = 0.500')

1000 applications, 31.0% female

  personal_status dropped                adversary AUC = 0.688
  chance                                 adversary AUC = 0.500


## Read the numbers

On Adult, removing the protected attribute and then the four features most
obviously associated with it still leaves the adversary comfortably above
chance. The single largest drop comes from `relationship`, and it comes from
there because that column *is* sex, not because the method found a subtle proxy.

German Credit is the smaller and more uncomfortable case. Sex is not a column.
It is a substring inside a categorical variable about marital status. A proxy
removal process driven by a list of column names does not see it, and neither
does a data protection impact assessment written from a schema.

The synthetic version in notebook 1 gets the same answer with tighter control:
there I know exactly how much gender information went in, so I can say what
fraction survived. Here I cannot, but I also did not build the data.

Use both. The simulation is for measuring the method against known ground truth.
The real benchmark is for checking the method was not measured against a world I
invented.